# Notebook 6: PropTech Air Quality Score
## *Rank addresses by pollution exposure — in seconds, at scale*

**Target audience:** PropTech platforms, real estate portals, insurance underwriters, mobility startups

**Use case:** Add air quality scores to property listings, insurance risk models, or delivery route rankings — without deploying sensors or scraping weather services.

### What we build
A reusable `air_score()` function that:
1. Takes a lat/lon address
2. Queries Jiskta for multi-year NO₂ + PM2.5
3. Returns a **0–100 air quality score** and a **WHO compliance tier**
4. Works across the whole CAMS dataset (Europe + global, 2013–present)

> **API cost:** 1 credit per query (covers a full year of data for a single point)

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from jiskta import JisktaClient

API_KEY = os.environ.get('JISKTA_API_KEY', 'your_api_key_here')
client = JisktaClient(API_KEY)

# WHO 2021 annual guidelines
WHO = {'no2': 10.0, 'pm2p5': 5.0, 'pm10': 15.0}

# EU Air Quality Directive (2024 revision, in force 2030)
EU_AQD_2030 = {'no2': 20.0, 'pm2p5': 10.0, 'pm10': 20.0}

print('Ready.')

## 1. The `air_score()` function

The score maps annual pollution exposure onto a **0–100 scale**:
- **80–100 (A):** Well below WHO 2021 guidelines — pristine air
- **60–79  (B):** Within WHO guidelines  
- **40–59  (C):** Below EU AQD 2030 limit (not yet WHO compliant)
- **20–39  (D):** Below current EU AQD limit (still legal, health concern)
- **0–19   (F):** Above current EU limits — regulatory risk

In [ ]:
def air_score(lat: float, lon: float, year: int = 2023,
              variables: list = ['no2', 'pm2p5']) -> dict:
    """
    Return air quality score (0-100) and WHO/AQD compliance tier for a location.
    Snaps to the nearest 0.1° CAMS grid cell (~11 km resolution).
    Cost: 1 credit per variable queried.
    """
    df = client.query(
        lat=lat, lon=lon,
        start=f'{year}-01', end=f'{year}-12',
        variables=variables,
        aggregate='monthly',
    )

    annual = {}
    for var in variables:
        col = f'{var}_mean'
        if col in df.columns:
            annual[var] = df[col].mean()

    # Score each pollutant: WHO guideline = 100 points, 4× WHO = 0 points
    scores = []
    for var, mean in annual.items():
        who_limit = WHO.get(var, None)
        if who_limit:
            raw = max(0, 1 - (mean - who_limit) / (4 * who_limit))
            scores.append(min(100, raw * 100))

    composite = np.mean(scores) if scores else None

    # Determine tier
    if composite is None:
        tier, color = 'N/A', '#999999'
    elif composite >= 80:
        tier, color = 'A — Excellent', '#16A34A'
    elif composite >= 60:
        tier, color = 'B — Good (WHO)', '#65A30D'
    elif composite >= 40:
        tier, color = 'C — Moderate (EU 2030)', '#CA8A04'
    elif composite >= 20:
        tier, color = 'D — Fair (EU current)', '#EA580C'
    else:
        tier, color = 'F — Poor', '#DC2626'

    return {
        'lat': lat, 'lon': lon, 'year': year,
        'annual': annual,
        'score': round(composite, 1) if composite else None,
        'tier': tier,
        'color': color,
    }


# Quick test
result = air_score(48.85, 2.35)  # Central Paris
print(f"Central Paris 2023: score={result['score']}, tier={result['tier']}")
print(f"  NO₂ annual: {result['annual'].get('no2', 'n/a'):.2f} µg/m³  (WHO limit: {WHO['no2']} µg/m³)")
print(f"  PM2.5 annual: {result['annual'].get('pm2p5', 'n/a'):.2f} µg/m³  (WHO limit: {WHO['pm2p5']} µg/m³)")

## 2. Score a portfolio of properties

Real estate portals can score thousands of listings per minute using the Jiskta API.
Here we score 12 Paris neighbourhoods across the price spectrum.

In [ ]:
# Paris neighbourhoods: (arrondissement/area, lat, lon)
properties = [
    ('Bois de Vincennes (12e)',    48.833, 2.430),
    ('Bois de Boulogne (16e)',     48.863, 2.245),
    ('Montmartre (18e)',           48.888, 2.342),
    ('Le Marais (4e)',             48.856, 2.354),
    ('Saint-Germain (6e)',         48.852, 2.330),
    ('Châtelet (1er)',             48.860, 2.347),
    ('République (11e)',           48.867, 2.362),
    ('Gare du Nord (10e)',         48.880, 2.355),
    ('La Défense',                 48.892, 2.237),
    ('Bagnolet (suburb)',          48.870, 2.420),
    ('Vincennes (suburb)',         48.848, 2.439),
    ('Versailles (suburb)',        48.805, 2.130),
]

scores = []
for name, lat, lon in properties:
    r = air_score(lat, lon, year=2023)
    scores.append({'Location': name, 'Score': r['score'],
                   'Tier': r['tier'], 'Color': r['color'],
                   'NO₂ µg/m³': round(r['annual'].get('no2', 0), 2),
                   'PM2.5 µg/m³': round(r['annual'].get('pm2p5', 0), 2)})
    print(f"  {name:<32} score={r['score']:>5}  {r['tier']}")

df_scores = pd.DataFrame(scores).sort_values('Score', ascending=False)
print(f'\nCost: {len(properties) * 2} credits (2 variables × {len(properties)} locations)')

## 3. Visualise the ranking

In [ ]:
fig, ax = plt.subplots(figsize=(11, 6))

bars = ax.barh(df_scores['Location'], df_scores['Score'],
               color=df_scores['Color'], edgecolor='white', linewidth=0.5)

# Score labels
for bar, score in zip(bars, df_scores['Score']):
    ax.text(bar.get_width() + 0.5, bar.get_y() + bar.get_height()/2,
            f'{score:.0f}', va='center', fontsize=9, fontweight='bold')

# Reference lines
ax.axvline(80, color='#16A34A', linestyle='--', alpha=0.5, linewidth=1)
ax.axvline(60, color='#CA8A04', linestyle='--', alpha=0.5, linewidth=1)
ax.text(80.5, ax.get_ylim()[1], 'WHO\n2021', fontsize=7, color='#16A34A', va='top')
ax.text(60.5, ax.get_ylim()[1], 'EU AQD\n2030', fontsize=7, color='#CA8A04', va='top')

ax.set_xlabel('Air Quality Score (0–100, higher = cleaner)', fontsize=11)
ax.set_title('Paris Neighbourhood Air Quality Scores — 2023\n'
             'Based on CAMS reanalysis via Jiskta API (NO₂ + PM2.5)', fontsize=13)
ax.set_xlim(0, 105)
ax.grid(axis='x', alpha=0.2)
ax.invert_yaxis()

legend_patches = [
    mpatches.Patch(color='#16A34A', label='A: Excellent (≥80)'),
    mpatches.Patch(color='#65A30D', label='B: Good / WHO (≥60)'),
    mpatches.Patch(color='#CA8A04', label='C: Moderate / EU 2030 (≥40)'),
    mpatches.Patch(color='#EA580C', label='D: Fair / EU current (≥20)'),
]
ax.legend(handles=legend_patches, fontsize=8, loc='lower right')
plt.tight_layout()
plt.savefig('../exports/06_proptech_scores.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. Multi-year trend — is the neighbourhood improving?

Buyers and tenants care about trajectory, not just today's score.
One API call per year (1 credit) gives a 10-year trend.

In [ ]:
# Compare two locations: suburban park vs inner-city
locations = {
    'Bois de Vincennes': (48.833, 2.430),
    'Gare du Nord':      (48.880, 2.355),
}
years = list(range(2015, 2024))

trends = {loc: [] for loc in locations}

for year in years:
    for loc, (lat, lon) in locations.items():
        df = client.query(lat=lat, lon=lon,
                          start=f'{year}-01', end=f'{year}-12',
                          variables=['no2'], aggregate='monthly')
        annual_mean = df['no2_mean'].mean()
        trends[loc].append(annual_mean)
    print(f'{year}: {" | ".join(f"{loc}: {trends[loc][-1]:.2f}" for loc in locations)}')

print(f'\nTotal cost: {len(years) * len(locations)} credits')

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
colors = {'Bois de Vincennes': '#16A34A', 'Gare du Nord': '#DC2626'}

for loc, vals in trends.items():
    ax.plot(years, vals, 'o-', label=loc, color=colors[loc], linewidth=2.5, markersize=7)

    # Trend line
    z = np.polyfit(years, vals, 1)
    p = np.poly1d(z)
    ax.plot(years, p(years), '--', color=colors[loc], alpha=0.4, linewidth=1)
    ax.text(years[-1] + 0.1, vals[-1],
            f'{z[0]:+.2f} µg/yr', fontsize=8, color=colors[loc], va='center')

ax.axhline(WHO['no2'], color='#16A34A', linestyle=':', linewidth=1.5,
           label=f'WHO 2021 guideline ({WHO["no2"]} µg/m³)')
ax.fill_between(years, 0, WHO['no2'], alpha=0.05, color='#16A34A')

ax.set_xlabel('Year')
ax.set_ylabel('Annual mean NO₂ (µg/m³)')
ax.set_title('NO₂ Trend 2015–2023: Suburban Park vs Inner City\n'
             'Dashed lines = linear trend, annotation = slope (µg/m³ per year)', fontsize=12)
ax.legend(fontsize=9)
ax.grid(alpha=0.2)
ax.set_xlim(years[0] - 0.3, years[-1] + 1.5)
plt.tight_layout()
plt.savefig('../exports/06_trend.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. Build an air quality API microservice

This is how you'd integrate Jiskta into a property platform backend:

```python
# Your platform's endpoint  →  calls Jiskta  →  returns enriched listing
from flask import Flask, request, jsonify
from jiskta import JisktaClient
import os

app = Flask(__name__)
client = JisktaClient(os.environ['JISKTA_API_KEY'])

@app.route('/api/listings/<listing_id>/air-quality')
def air_quality(listing_id):
    lat  = float(request.args['lat'])
    lon  = float(request.args['lon'])
    year = int(request.args.get('year', 2023))

    result = air_score(lat, lon, year=year)  # 1 credit

    return jsonify({
        'listing_id': listing_id,
        'air_quality_score': result['score'],
        'tier': result['tier'],
        'no2_annual_mean':   result['annual'].get('no2'),
        'pm25_annual_mean':  result['annual'].get('pm2p5'),
        'who_no2_limit':     10.0,
        'data_source':       'CAMS reanalysis via Jiskta',
    })
```

**Cost structure for a PropTech platform:**
| Scale | Queries/month | Cost (Jiskta) |
|-------|--------------|---------------|
| MVP   | 1,000        | €1.54          |
| Growth| 50,000       | €71            |
| Scale | 500,000      | €667           |